## INSTALL LIBRARIES (CELL 1)

In [1]:
!pip install semantic-link-sempy semantic-link-labs
!pip install --upgrade semantic-link --q
!pip install notebookutils

StatementMeta(, 5e224871-1db6-41c5-890a-39168d08124b, 3, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 kB 8.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 66.7 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 2024.9.1
    Uninstalling azure-core-2024.9.1:
      Successfully uninstalled azure-core-2024.9.1
  Attempting uninstall: semantic-link-sempy
    Found existing installation: semantic-link-sempy 0.11.0
    Uninstalling semantic-link-sempy-0.11.0:
      Successfully

## IMPORT LIBRARIES ( CELL 2 )

In [2]:
import os
import re
import json
import time
import tempfile
import posixpath
import traceback
import requests
import pandas as pd
import sempy.fabric as fabric
import random
from collections import deque
from typing import Dict, List, Optional, Tuple

from datetime import datetime, timezone
import notebookutils
from sempy.fabric import FabricRestClient

StatementMeta(, 5e224871-1db6-41c5-890a-39168d08124b, 4, Finished, Available, Finished, False)

## SETTING PARAMETERS ( CELL 3 )

In [ ]:

# =========================================================
# RESTORE PARAMETERS
# =========================================================

# ===== ADLS backup source =====
# Storage / backup location
STORAGE_ACCOUNT = "stor*****"
FILE_SYSTEM = "fabric-backup"
BACKUP_ROOT = "fabric-export/2026-04-22T13-54-41Z"   # top-level export folder under the filesystem/container

# Exact workspace backup folder created by the backup script
# Example: "workspace1__ae4d10c3-b9bf-45e8-892f-4203aec2fdc8"
SOURCE_WORKSPACE_NAME = "test_move_1"

# New target workspace name to create
TARGET_WORKSPACE_NAME = "test_move_1-restored-test-02"

# ===== restore behavior =====
RESTORE_WORKSPACE_ACCESS = True
RESTORE_DATASET_PERMISSIONS = True
RESTORE_RLS_ROLE_MEMBERS = True
RESTORE_WORKSPACE_DESCRIPTION = True

# Use original workspace values if present in backup
RESTORE_CAPACITY_ID = True
RESTORE_DOMAIN_ID = True

# Restore generic definition-based item types beyond
# SemanticModel / Report / PaginatedReport
RESTORE_OTHER_DEFINITION_ITEMS = True

# Stop after first major failure
FAIL_FAST = False

# ===== API throttling =====
ENABLE_CLIENT_THROTTLING = True

# Conservative pacing to avoid bursts
MAX_REQUESTS_PER_MINUTE = 90
MIN_SECONDS_BETWEEN_CALLS = 0.75
THROTTLE_JITTER_SECONDS = 0.20

ABFSS_BASE = f"abfss://{FILE_SYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{BACKUP_ROOT}"
RESTORE_RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
RESTORE_LOG_ROOT = f"{ABFSS_BASE}/_restore_runs/{RESTORE_RUN_ID}"

print("Backup source root              :", ABFSS_BASE)
print("Restore log root                :", RESTORE_LOG_ROOT)
print("Source workspace                :", SOURCE_WORKSPACE_NAME)
print("Target workspace                :", TARGET_WORKSPACE_NAME)
print("Client-side throttling enabled  :", ENABLE_CLIENT_THROTTLING)
print("Max requests per minute         :", MAX_REQUESTS_PER_MINUTE)
print("Min seconds between calls       :", MIN_SECONDS_BETWEEN_CALLS)
print("Throttle jitter seconds         :", THROTTLE_JITTER_SECONDS)

# Restore behavior flags
USE_BACKED_UP_WORKSPACE_DESCRIPTION = True
RESTORE_WORKSPACE_ACCESS = True
RESTORE_ITEM_DEFINITIONS = True
RESTORE_PAGINATED_REPORTS = True
RESTORE_SEMANTIC_MODEL_RLS_MEMBERS = True
UPSERT_IF_ITEM_EXISTS = True

# Retry / polling
MAX_RETRIES = 5
DEFAULT_RETRY_SECONDS = 10
IMPORT_POLL_SECONDS = 5
IMPORT_TIMEOUT_SECONDS = 600

# Secret names in Key Vault
TENANT_ID_SECRET = "94***c6"
CLIENT_ID_SECRET = "0b***4df"
CLIENT_SECRET_SECRET = "Pu***cvT"


## AZURE DATA LAKE CONNECTION SETTINGS ( CELL 4 )

In [ ]:
tenant_id = TENANT_ID_SECRET
client_id = CLIENT_ID_SECRET
client_secret = CLIENT_SECRET_SECRET

account_fqdn = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.conf.set(f"fs.azure.account.auth.type.{account_fqdn}", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{account_fqdn}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("ABFSS OAuth configuration applied.")

## DEFINE HELPER FUNCTIONS ( CELL 5 )

In [5]:
# =========================================================
# HELPER FUNCTIONS
# =========================================================

rest_client = FabricRestClient()

# ---------------------------------------------------------
# basic helpers
# ---------------------------------------------------------

def sanitize_name(value: str) -> str:
    value = str(value or "").strip()
    value = re.sub(r'[<>:"/\\\\|?*]+', "_", value)
    value = re.sub(r"\s+", " ", value)
    return value[:180] if value else "unknown"

def ensure_folder(path: str):
    if not notebookutils.fs.exists(path):
        notebookutils.fs.mkdirs(path)

def write_text(path: str, text: str, overwrite: bool = True):
    parent = path.rsplit("/", 1)[0]
    ensure_folder(parent)
    notebookutils.fs.put(path, text, overwrite)

def path_exists(path: str) -> bool:
    return notebookutils.fs.exists(path)

def exists(path: str) -> bool:
    return notebookutils.fs.exists(path)

def ls(path: str):
    return notebookutils.fs.ls(path)

def log_message(message: str):
    ts = datetime.now(timezone.utc).isoformat()
    print(f"[{ts}] {message}")

def write_restore_artifact(relative_path: str, payload):
    path = f"{RESTORE_LOG_ROOT}/{relative_path}"
    if isinstance(payload, str):
        write_text(path, payload, overwrite=True)
    else:
        write_text(path, json.dumps(payload, indent=2, ensure_ascii=False, default=str), overwrite=True)

def raise_if_fail_fast(ex: Exception):
    if FAIL_FAST:
        raise ex

# ---------------------------------------------------------
# safer file reading
# ---------------------------------------------------------

def read_text(path: str) -> str:
    if not notebookutils.fs.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    local_tmp = f"/tmp/{int(time.time() * 1000)}_{os.path.basename(path)}"
    notebookutils.fs.cp(path, f"file:{local_tmp}")

    with open(local_tmp, "r", encoding="utf-8-sig") as f:
        content = f.read()

    return content

def read_json(path: str):
    content = read_text(path).strip()

    if not content:
        raise ValueError(f"JSON file is empty: {path}")

    try:
        return json.loads(content)
    except Exception as ex:
        preview = content[:500]
        raise ValueError(
            f"Failed to parse JSON file: {path}\n"
            f"First 500 chars:\n{preview}"
        ) from ex

def get_file_bytes(path: str) -> bytes:
    local_tmp = f"/tmp/{int(time.time() * 1000)}_{os.path.basename(path)}"
    notebookutils.fs.cp(path, f"file:{local_tmp}")
    with open(local_tmp, "rb") as f:
        return f.read()

# ---------------------------------------------------------
# throttling
# ---------------------------------------------------------

_api_call_timestamps = deque()
_last_api_call_ts = 0.0

def throttle_api_calls(
    max_requests_per_minute: int = MAX_REQUESTS_PER_MINUTE,
    min_seconds_between_calls: float = MIN_SECONDS_BETWEEN_CALLS,
    jitter_seconds: float = THROTTLE_JITTER_SECONDS
):
    global _api_call_timestamps, _last_api_call_ts

    if not ENABLE_CLIENT_THROTTLING:
        return

    now = time.time()

    while _api_call_timestamps and (now - _api_call_timestamps[0]) >= 60:
        _api_call_timestamps.popleft()

    if len(_api_call_timestamps) >= max_requests_per_minute:
        sleep_for = 60 - (now - _api_call_timestamps[0]) + random.uniform(0, jitter_seconds)
        sleep_for = max(sleep_for, 0.0)
        log_message(
            f"Client throttling engaged: {len(_api_call_timestamps)} requests in rolling 60s window. "
            f"Sleeping {sleep_for:.2f}s"
        )
        time.sleep(sleep_for)

        now = time.time()
        while _api_call_timestamps and (now - _api_call_timestamps[0]) >= 60:
            _api_call_timestamps.popleft()

    now = time.time()
    elapsed_since_last = now - _last_api_call_ts
    if elapsed_since_last < min_seconds_between_calls:
        sleep_for = (min_seconds_between_calls - elapsed_since_last) + random.uniform(0, jitter_seconds)
        sleep_for = max(sleep_for, 0.0)
        if sleep_for > 0:
            time.sleep(sleep_for)

    _last_api_call_ts = time.time()
    _api_call_timestamps.append(_last_api_call_ts)

# ---------------------------------------------------------
# auth headers
# ---------------------------------------------------------

def get_pbi_access_token() -> str:
    return notebookutils.credentials.getToken("pbi")

def pbi_headers(extra: Optional[dict] = None) -> dict:
    headers = {
        "Authorization": f"Bearer {get_pbi_access_token()}"
    }
    if extra:
        headers.update(extra)
    return headers

# ---------------------------------------------------------
# HTTP request helpers
# ---------------------------------------------------------

def pbi_request_with_retry(method: str, url: str, **kwargs):
    max_retries = 8
    default_retry_seconds = 15

    for attempt in range(1, max_retries + 1):
        throttle_api_calls()

        resp = requests.request(method, url, **kwargs)

        if resp.status_code in (200, 201, 202):
            return resp

        if resp.status_code == 429 or resp.status_code >= 500:
            retry_after = int(resp.headers.get("Retry-After", default_retry_seconds))
            retry_after = retry_after + random.uniform(0, THROTTLE_JITTER_SECONDS)

            log_message(
                f"PBI retryable response {resp.status_code}. "
                f"Sleeping {retry_after:.2f}s ({attempt}/{max_retries})"
            )
            time.sleep(retry_after)
            continue

        raise RuntimeError(f"PBI API {resp.status_code}: {resp.text}")

    raise RuntimeError(f"PBI API failed after {max_retries} retries: {method} {url}")

def fabric_request_with_retry(method: str, path: str, **kwargs):
    max_retries = 8
    default_retry_seconds = 10

    for attempt in range(1, max_retries + 1):
        throttle_api_calls()

        response = getattr(rest_client, method)(path, **kwargs)

        if response.status_code in (200, 201, 202):
            return response

        if response.status_code == 429 or response.status_code >= 500:
            retry_after = int(response.headers.get("Retry-After", default_retry_seconds))
            retry_after = retry_after + random.uniform(0, THROTTLE_JITTER_SECONDS)

            log_message(
                f"Fabric retryable response {response.status_code}. "
                f"Sleeping {retry_after:.2f}s ({attempt}/{max_retries})"
            )
            time.sleep(retry_after)
            continue

        raise RuntimeError(f"Fabric API {response.status_code}: {response.text}")

    raise RuntimeError(f"Fabric API failed after {max_retries} retries: {method.upper()} {path}")

# ---------------------------------------------------------
# long-running ops
# ---------------------------------------------------------

def wait_for_fabric_lro(response):
    if response.status_code in (200, 201):
        return response.json() if response.text else {}

    if response.status_code != 202:
        raise RuntimeError(f"Unexpected Fabric LRO status: {response.status_code} {response.text}")

    operation_id = response.headers.get("x-ms-operation-id")
    if not operation_id:
        raise RuntimeError("Fabric LRO started but x-ms-operation-id header missing.")

    while True:
        op_resp = fabric_request_with_retry("get", f"/v1/operations/{operation_id}")
        op_json = op_resp.json()
        status = op_json.get("status")

        if status == "Succeeded":
            result_resp = rest_client.get(f"/v1/operations/{operation_id}/result")
            if result_resp.status_code == 200 and result_resp.text:
                return result_resp.json()
            return {}

        if status in ("Failed", "Canceled"):
            raise RuntimeError(f"Fabric LRO failed: {json.dumps(op_json, indent=2)}")

        retry_after = int(op_resp.headers.get("Retry-After", 10))
        time.sleep(retry_after)

def wait_for_import_in_group(workspace_id: str, import_id: str) -> dict:
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/imports/{import_id}"

    while True:
        resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
        payload = resp.json()
        state = str(payload.get("importState", "")).lower()

        if state in ("succeeded", "failed"):
            return payload

        time.sleep(10)

# ---------------------------------------------------------
# workspace backup discovery
# ---------------------------------------------------------

def discover_workspace_folder() -> str:
    manifest = read_json(f"{ABFSS_BASE}/_manifest.json")

    candidates = [
        r for r in manifest
        if r.get("workspace_name") == SOURCE_WORKSPACE_NAME
    ]

    if not candidates:
        raise ValueError(f"No rows found in _manifest.json for workspace '{SOURCE_WORKSPACE_NAME}'")

    workspace_ids = sorted(set([r["workspace_id"] for r in candidates if r.get("workspace_id")]))
    if len(workspace_ids) != 1:
        raise ValueError(f"Expected one workspace id for '{SOURCE_WORKSPACE_NAME}', found: {workspace_ids}")

    workspace_id = workspace_ids[0]
    folder_name = f"{sanitize_name(SOURCE_WORKSPACE_NAME)}__{workspace_id}"
    folder_path = f"{ABFSS_BASE}/{folder_name}"

    if not exists(folder_path):
        raise FileNotFoundError(f"Workspace backup folder not found: {folder_path}")

    return folder_path

def read_workspace_backup_context() -> dict:
    workspace_folder = discover_workspace_folder()
    workspace_json_path = f"{workspace_folder}/_workspace.json"
    access_json_path = f"{workspace_folder}/_workspace_access_role_assignments.json"
    dashboards_json_path = f"{workspace_folder}/_dashboards_inventory.json"
    manifest_path = f"{ABFSS_BASE}/_manifest.json"

    manifest = pd.DataFrame(read_json(manifest_path))
    manifest = manifest[manifest["workspace_name"] == SOURCE_WORKSPACE_NAME].copy()

    workspace_json = read_json(workspace_json_path) if exists(workspace_json_path) else {}
    workspace_access = read_json(access_json_path) if exists(access_json_path) else []
    dashboard_inventory = read_json(dashboards_json_path) if exists(dashboards_json_path) else []

    return {
        "workspace_folder": workspace_folder,
        "workspace_json": workspace_json,
        "workspace_access": workspace_access,
        "dashboard_inventory": dashboard_inventory,
        "manifest_df": manifest
    }

# ---------------------------------------------------------
# item backup helpers
# ---------------------------------------------------------

def get_definition_from_backup(item_base_folder: str) -> dict:
    path = f"{item_base_folder}/_definition_response.json"
    definition_response = read_json(path)
    definition = definition_response.get("definition")
    if not isinstance(definition, dict):
        raise ValueError(f"No 'definition' object found in {path}")
    return definition

def get_paginated_rdl_path(item_base_folder: str) -> str:
    entries = ls(item_base_folder)
    rdl_files = [x.path for x in entries if x.path.lower().endswith(".rdl")]
    if not rdl_files:
        raise FileNotFoundError(f"No .rdl file found under {item_base_folder}")
    return rdl_files[0]

def get_dataflow_gen1_model_json_path(item_base_folder: str) -> str:
    """
    Expected backup layout for Dataflow Gen1:
      <workspace_folder>/DataflowGen1/<item_name>__<item_id>/model.json

    metadata.json / datasources.json / transactions.json / upstreamDataflows.json
    may also exist, but model.json is the core restore artifact.
    """
    candidate = f"{item_base_folder}/model.json"
    if exists(candidate):
        return candidate

    entries = ls(item_base_folder)
    json_files = [x.path for x in entries if x.path.lower().endswith("/model.json") or x.path.lower().endswith("model.json")]
    if json_files:
        return json_files[0]

    raise FileNotFoundError(f"No model.json found for Dataflow Gen1 under {item_base_folder}")

def get_optional_json_if_exists(path: str):
    return read_json(path) if exists(path) else None

# ---------------------------------------------------------
# workspace create / access restore
# ---------------------------------------------------------

def create_workspace_from_backup(workspace_json: dict) -> dict:
    body = {
        "displayName": TARGET_WORKSPACE_NAME
    }

    if RESTORE_WORKSPACE_DESCRIPTION and workspace_json.get("description"):
        body["description"] = workspace_json.get("description")

    if RESTORE_CAPACITY_ID and workspace_json.get("capacityId"):
        body["capacityId"] = workspace_json.get("capacityId")

    if RESTORE_DOMAIN_ID and workspace_json.get("domainId"):
        body["domainId"] = workspace_json.get("domainId")

    log_message(f"Creating target workspace '{TARGET_WORKSPACE_NAME}'")
    resp = fabric_request_with_retry("post", "/v1/workspaces", json=body)
    payload = resp.json()

    workspace_id = payload["id"]

    patch_body = {"displayName": TARGET_WORKSPACE_NAME}
    if RESTORE_WORKSPACE_DESCRIPTION:
        patch_body["description"] = workspace_json.get("description", "")

    fabric_request_with_retry("patch", f"/v1/workspaces/{workspace_id}", json=patch_body)

    return payload

def list_workspace_role_assignments(workspace_id: str) -> list:
    all_rows = []
    continuation_token = None

    while True:
        path = f"/v1/workspaces/{workspace_id}/roleAssignments"
        if continuation_token:
            path += f"?continuationToken={continuation_token}"

        resp = fabric_request_with_retry("get", path)
        payload = resp.json() if resp.text else {}

        rows = payload.get("value", [])
        all_rows.extend(rows)

        continuation_token = payload.get("continuationToken")
        if not continuation_token:
            break

    return all_rows

def principal_already_exists(existing_assignments: list, principal_id: str) -> Optional[dict]:
    for ra in existing_assignments:
        principal = ra.get("principal", {}) or {}
        if str(principal.get("id", "")).lower() == str(principal_id).lower():
            return ra
    return None

def add_workspace_role_assignment_safe(workspace_id: str, assignment: dict, existing_assignments: list) -> str:
    principal = assignment.get("principal", {}) or {}
    role = assignment.get("role")

    if not principal or not role:
        raise ValueError(f"Invalid role assignment payload: {assignment}")

    principal_id = principal.get("id")
    principal_type = principal.get("type")

    if not principal_id or not principal_type:
        raise ValueError(f"Missing principal id/type in role assignment: {assignment}")

    existing = principal_already_exists(existing_assignments, principal_id)
    if existing:
        existing_role = existing.get("role")
        if str(existing_role).lower() == str(role).lower():
            log_message(
                f"Workspace access already exists, skipping: "
                f"{principal.get('displayName', principal_id)} ({existing_role})"
            )
            return "skipped_exists_same_role"
        else:
            log_message(
                f"Workspace principal already exists with different role, skipping: "
                f"{principal.get('displayName', principal_id)} "
                f"(existing={existing_role}, requested={role})"
            )
            return "skipped_exists_different_role"

    body = {
        "principal": {
            "id": principal_id,
            "type": principal_type
        },
        "role": role
    }

    try:
        fabric_request_with_retry("post", f"/v1/workspaces/{workspace_id}/roleAssignments", json=body)
        existing_assignments.append({
            "principal": {
                "id": principal_id,
                "type": principal_type,
                "displayName": principal.get("displayName", "")
            },
            "role": role
        })
        log_message(f"Restored workspace role assignment: {principal.get('displayName', principal_id)} -> {role}")
        return "restored"

    except Exception as ex:
        msg = str(ex)
        if "PrincipalAlreadyHasWorkspaceRolePermissions" in msg or "409 Conflict" in msg:
            log_message(
                f"Workspace access already assigned by service, skipping: "
                f"{principal.get('displayName', principal_id)}"
            )
            return "skipped_exists_same_role"
        raise

def restore_workspace_access(workspace_id: str, workspace_access: list) -> Tuple[int, int, int]:
    restored = 0
    skipped = 0
    errors = 0

    existing_assignments = list_workspace_role_assignments(workspace_id)

    for ra in workspace_access:
        try:
            principal = ra.get("principal", {}) or {}
            principal_type = principal.get("type", "")

            if principal_type == "EntireTenant":
                skipped += 1
                log_message("Skipped workspace access restore for EntireTenant principal.")
                continue

            result = add_workspace_role_assignment_safe(
                workspace_id=workspace_id,
                assignment=ra,
                existing_assignments=existing_assignments
            )

            if result == "restored":
                restored += 1
            else:
                skipped += 1

        except Exception as ex:
            errors += 1
            log_message(f"Workspace access restore error: {ex}")

    return restored, skipped, errors

# ---------------------------------------------------------
# item creation helpers
# ---------------------------------------------------------

def create_item_generic(workspace_id: str, item_type: str, display_name: str, definition: dict, description: str = "") -> dict:
    body = {
        "displayName": display_name,
        "type": item_type,
        "definition": definition
    }
    if description:
        body["description"] = description[:256]

    resp = fabric_request_with_retry("post", f"/v1/workspaces/{workspace_id}/items", json=body)
    payload = wait_for_fabric_lro(resp)
    return payload

def create_semantic_model(workspace_id: str, display_name: str, definition: dict, description: str = "") -> dict:
    body = {
        "displayName": display_name,
        "definition": definition
    }
    if description:
        body["description"] = description[:256]

    resp = fabric_request_with_retry("post", f"/v1/workspaces/{workspace_id}/semanticModels", json=body)
    payload = wait_for_fabric_lro(resp)
    return payload

def create_report(workspace_id: str, display_name: str, definition: dict, description: str = "") -> dict:
    body = {
        "displayName": display_name,
        "definition": definition
    }
    if description:
        body["description"] = description[:256]

    resp = fabric_request_with_retry("post", f"/v1/workspaces/{workspace_id}/reports", json=body)
    payload = wait_for_fabric_lro(resp)
    return payload

def import_paginated_report(workspace_id: str, display_name: str, rdl_bytes: bytes) -> dict:
    dataset_display_name = display_name if display_name.lower().endswith(".rdl") else f"{display_name}.rdl"
    url = (
        f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/imports"
        f"?datasetDisplayName={requests.utils.quote(dataset_display_name)}&nameConflict=Abort"
    )

    resp = pbi_request_with_retry(
        "POST",
        url,
        headers=pbi_headers(),
        files={
            "file": (dataset_display_name, rdl_bytes, "application/rdl")
        }
    )

    payload = resp.json()
    import_id = payload["id"]
    final_payload = wait_for_import_in_group(workspace_id, import_id)

    if str(final_payload.get("importState", "")).lower() != "succeeded":
        raise RuntimeError(f"Paginated report import failed: {json.dumps(final_payload, indent=2)}")

    return final_payload

def import_dataflow_gen1(workspace_id: str, display_name: str, model_json_bytes: bytes) -> dict:
    """
    Restores Power BI / Fabric Dataflow Gen1 by importing model.json
    through the Power BI Imports API.
    """
    url = (
        f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/imports"
        f"?datasetDisplayName=model.json&nameConflict=Abort"
    )

    resp = pbi_request_with_retry(
        "POST",
        url,
        headers=pbi_headers(),
        files={
            "file": ("model.json", model_json_bytes, "application/json")
        }
    )

    payload = resp.json()
    import_id = payload["id"]
    final_payload = wait_for_import_in_group(workspace_id, import_id)

    if str(final_payload.get("importState", "")).lower() != "succeeded":
        raise RuntimeError(f"Dataflow Gen1 import failed: {json.dumps(final_payload, indent=2)}")

    return final_payload

# ---------------------------------------------------------
# dataset permissions
# ---------------------------------------------------------

def normalize_dataset_permission_record(rec: dict) -> dict:
    return {
        "identifier": rec.get("identifier") or rec.get("emailAddress") or rec.get("graphId"),
        "principalType": rec.get("principalType"),
        "datasetUserAccessRight": rec.get("datasetUserAccessRight")
    }

def apply_dataset_permissions(workspace_id: str, dataset_id: str, item_base_folder: str) -> Tuple[int, int]:
    path = f"{item_base_folder}/_dataset_permissions.json"
    if not exists(path):
        log_message(f"No dataset permissions file found for semantic model at {item_base_folder}")
        return 0, 0

    rows = read_json(path)
    applied = 0
    skipped = 0

    for rec in rows:
        try:
            payload = normalize_dataset_permission_record(rec)

            if not payload["identifier"] or not payload["principalType"] or not payload["datasetUserAccessRight"]:
                skipped += 1
                continue

            if str(payload["principalType"]).lower() == "app":
                skipped += 1
                log_message(f"Skipped dataset permission for service principal: {payload['identifier']}")
                continue

            url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/users"

            try:
                pbi_request_with_retry("POST", url, headers=pbi_headers({"Content-Type": "application/json"}), json=payload)
            except Exception:
                pbi_request_with_retry("PUT", url, headers=pbi_headers({"Content-Type": "application/json"}), json=payload)

            applied += 1
            log_message(f"Applied dataset permission: {payload['identifier']} -> {payload['datasetUserAccessRight']}")
        except Exception as ex:
            skipped += 1
            log_message(f"Skipped dataset permission row due to error: {ex}")

    return applied, skipped

# ---------------------------------------------------------
# RLS helpers
# ---------------------------------------------------------

def _find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_map = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_map:
            return cols_map[c.lower()]
    return None

def get_backup_rls_role_summary(item_folder_path: str) -> dict:
    roles_path = f"{item_folder_path}/_roles_with_members.json"
    if not path_exists(roles_path):
        return {
            "has_backup": False,
            "status": "no_roles_backup",
            "roles": [],
            "rows": 0
        }

    rows = read_json(roles_path)
    df = pd.DataFrame(rows)
    if df.empty:
        return {
            "has_backup": True,
            "status": "empty_roles_backup",
            "roles": [],
            "rows": 0
        }

    role_col = _find_col(df, ["Role Name", "role_name", "role"])
    if not role_col:
        return {
            "has_backup": True,
            "status": "unrecognized_roles_schema",
            "roles": [],
            "rows": len(df)
        }

    df[role_col] = df[role_col].astype(str).str.strip()
    role_names = sorted([r for r in df[role_col].dropna().unique().tolist() if str(r).strip()])

    return {
        "has_backup": True,
        "status": "ok",
        "roles": role_names,
        "rows": len(df)
    }

def inspect_restored_semantic_model_roles(workspace_id: str, semantic_model_id: str) -> dict:
    role_names = []
    role_member_map = {}

    try:
        throttle_api_calls()

        with fabric.connect_semantic_model(
            dataset=semantic_model_id,
            workspace=workspace_id,
            readonly=True
        ) as tom:

            model = getattr(tom, "model", None) or getattr(tom, "Model", None)
            if model is None:
                return {
                    "status": "error_no_model",
                    "roles": [],
                    "role_member_map": {}
                }

            for role_obj in model.Roles:
                role_name = str(getattr(role_obj, "Name", "")).strip()
                if not role_name:
                    continue

                role_names.append(role_name)

                members = []
                try:
                    for m in role_obj.Members:
                        if hasattr(m, "MemberName") and m.MemberName:
                            members.append(str(m.MemberName).strip())
                        elif hasattr(m, "Name") and m.Name:
                            members.append(str(m.Name).strip())
                except Exception:
                    pass

                role_member_map[role_name] = sorted(set([x for x in members if x]))

        return {
            "status": "ok",
            "roles": sorted(role_names),
            "role_member_map": role_member_map
        }

    except Exception as ex:
        return {
            "status": f"error: {ex}",
            "roles": [],
            "role_member_map": {}
        }

def restore_semantic_model_rls_members(
    workspace_id: str,
    semantic_model_id: str,
    semantic_model_name: str,
    item_folder_path: str,
    checkpoint_every_roles: int = 10
) -> dict:
    roles_path = f"{item_folder_path}/_roles_with_members.json"
    if not path_exists(roles_path):
        return {
            "status": "skipped_no_roles_backup",
            "restored": 0,
            "skipped": 0,
            "errors": 0,
            "missing_roles": []
        }

    roles_rows = read_json(roles_path)
    roles_df = pd.DataFrame(roles_rows)
    if roles_df.empty:
        return {
            "status": "skipped_empty_roles_backup",
            "restored": 0,
            "skipped": 0,
            "errors": 0,
            "missing_roles": []
        }

    role_col = _find_col(roles_df, ["Role Name", "role_name", "role"])
    member_col = _find_col(roles_df, ["Member Name", "member_name", "member"])
    member_type_col = _find_col(roles_df, ["Member Type", "member_type"])

    if not role_col or not member_col:
        return {
            "status": "skipped_unrecognized_roles_schema",
            "restored": 0,
            "skipped": 0,
            "errors": 1,
            "missing_roles": []
        }

    if member_type_col:
        roles_df = roles_df[
            ~roles_df[member_type_col].astype(str).str.lower().isin(["serviceprincipal", "service principal"])
        ].copy()

    if roles_df.empty:
        return {
            "status": "skipped_no_supported_role_members",
            "restored": 0,
            "skipped": 0,
            "errors": 0,
            "missing_roles": []
        }

    roles_df[role_col] = roles_df[role_col].astype(str).str.strip()
    roles_df[member_col] = roles_df[member_col].astype(str).str.strip()
    roles_df = roles_df[(roles_df[role_col] != "") & (roles_df[member_col] != "")].copy()

    if roles_df.empty:
        return {
            "status": "skipped_no_valid_role_members",
            "restored": 0,
            "skipped": 0,
            "errors": 0,
            "missing_roles": []
        }

    dedupe_cols = [role_col, member_col]
    if member_type_col:
        dedupe_cols.append(member_type_col)
    roles_df = roles_df.drop_duplicates(subset=dedupe_cols).copy()

    restored = 0
    skipped = 0
    errors = 0
    missing_roles = []
    changes_made = False

    try:
        import Microsoft.AnalysisServices.Tabular as TOM

        throttle_api_calls()

        with fabric.connect_semantic_model(
            dataset=semantic_model_id,
            workspace=workspace_id,
            readonly=False
        ) as tom:

            model = getattr(tom, "model", None) or getattr(tom, "Model", None)
            if model is None:
                raise RuntimeError("Could not access TOM model object from connect_semantic_model()")

            role_obj_map = {}
            existing_members_by_role = {}

            for role_obj in model.Roles:
                role_name = str(getattr(role_obj, "Name", "")).strip()
                if not role_name:
                    continue

                role_key = role_name.lower()
                role_obj_map[role_key] = role_obj

                members = set()
                try:
                    for m in role_obj.Members:
                        member_name = None

                        if hasattr(m, "MemberName") and m.MemberName:
                            member_name = str(m.MemberName).strip()
                        elif hasattr(m, "Name") and m.Name:
                            member_name = str(m.Name).strip()

                        if member_name:
                            members.add(member_name.lower())
                except Exception:
                    pass

                existing_members_by_role[role_key] = members

            grouped_roles = list(roles_df.groupby(role_col))

            for role_idx, (backup_role_name, grp) in enumerate(grouped_roles, start=1):
                backup_role_name = str(backup_role_name).strip()
                role_key = backup_role_name.lower()

                role_obj = role_obj_map.get(role_key)
                if role_obj is None:
                    missing_roles.append(backup_role_name)
                    errors += len(grp)
                    log_message(
                        f"RLS role from backup not found in restored model '{semantic_model_name}': {backup_role_name}. "
                        f"Members for this role cannot be restored."
                    )
                    continue

                existing_members = existing_members_by_role.setdefault(role_key, set())

                for _, r in grp.iterrows():
                    member_name = str(r.get(member_col, "")).strip()
                    if not member_name:
                        skipped += 1
                        continue

                    member_key = member_name.lower()
                    if member_key in existing_members:
                        skipped += 1
                        continue

                    raw_member_type = str(r.get(member_type_col, "User")).strip().lower() if member_type_col else "user"

                    try:
                        member_obj = TOM.ExternalModelRoleMember()

                        if hasattr(member_obj, "MemberName"):
                            member_obj.MemberName = member_name
                        elif hasattr(member_obj, "Name"):
                            member_obj.Name = member_name
                        else:
                            raise RuntimeError("ExternalModelRoleMember has no writable name property")

                        if hasattr(member_obj, "IdentityProvider"):
                            member_obj.IdentityProvider = "AzureAD"

                        if hasattr(member_obj, "RoleMemberType"):
                            if raw_member_type.startswith("group"):
                                try:
                                    member_obj.RoleMemberType = TOM.RoleMemberType.Group
                                except Exception:
                                    member_obj.RoleMemberType = "Group"
                            else:
                                try:
                                    member_obj.RoleMemberType = TOM.RoleMemberType.User
                                except Exception:
                                    member_obj.RoleMemberType = "User"

                        role_obj.Members.Add(member_obj)

                        existing_members.add(member_key)
                        restored += 1
                        changes_made = True

                        log_message(
                            f"Restored RLS member '{member_name}' to role '{backup_role_name}' "
                            f"for semantic model '{semantic_model_name}'"
                        )

                    except Exception as member_ex:
                        errors += 1
                        log_message(
                            f"Failed to restore RLS member '{member_name}' to role '{backup_role_name}' "
                            f"in semantic model '{semantic_model_name}': {member_ex}"
                        )

                if ENABLE_CLIENT_THROTTLING and role_idx % checkpoint_every_roles == 0:
                    throttle_api_calls()

            if changes_made:
                throttle_api_calls()

                try:
                    model.SaveChanges()
                except Exception:
                    save_meth = getattr(tom, "save_changes", None) or getattr(tom, "SaveChanges", None)
                    if save_meth:
                        save_meth()
                    else:
                        raise

    except Exception as ex:
        return {
            "status": f"error: {ex}",
            "restored": restored,
            "skipped": skipped,
            "errors": errors + 1,
            "missing_roles": sorted(set(missing_roles))
        }

    if errors > 0 and restored == 0:
        final_status = "warning_no_rls_members_restored"
    elif errors > 0:
        final_status = "warning_partial_rls_restore"
    elif restored == 0 and len(roles_df) > 0:
        final_status = "warning_no_new_rls_members_needed_or_applied"
    else:
        final_status = "completed"

    return {
        "status": final_status,
        "restored": restored,
        "skipped": skipped,
        "errors": errors,
        "missing_roles": sorted(set(missing_roles))
    }

# ---------------------------------------------------------
# sorting / output helpers
# ---------------------------------------------------------

def item_sort_bucket(item_type: str) -> int:
    order = {
        "Lakehouse": 10,
        "Warehouse": 20,
        "SQLEndpoint": 30,
        "Environment": 40,
        "Notebook": 50,
        "DataPipeline": 60,
        "Dataflow": 70,
        "DataflowGen1": 71,
        "SemanticModel": 100,
        "Report": 200,
        "PaginatedReport": 210,
        "Dashboard": 999
    }
    return order.get(item_type, 150)

def safe_value(v):
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    if isinstance(v, float) and math.isnan(v):
        return ""
    return v

StatementMeta(, 5e224871-1db6-41c5-890a-39168d08124b, 7, Finished, Available, Finished, False)

## DISCOVER BACK UP CONTENT ( CELL 6 )

In [ ]:
# =========================================================
# DISCOVER BACKUP CONTENT
# =========================================================

try:
    ctx = read_workspace_backup_context()

    workspace_folder = ctx["workspace_folder"]
    workspace_json = ctx["workspace_json"]
    workspace_access = ctx["workspace_access"]
    dashboard_inventory = ctx["dashboard_inventory"]
    manifest_df = ctx["manifest_df"].copy()

    display_name_from_backup = workspace_json.get("displayName", SOURCE_WORKSPACE_NAME)
    workspace_id_from_backup = workspace_json.get("id")

    log_message(f"Workspace backup folder located: {workspace_folder}")
    log_message(f"Workspace backup display name    : {display_name_from_backup}")
    log_message(f"Workspace backup id              : {workspace_id_from_backup}")
    log_message(f"Manifest item rows found         : {len(manifest_df)}")
    log_message(f"Dashboard inventory rows found   : {len(dashboard_inventory)}")

    if manifest_df.empty:
        raise ValueError("No manifest rows found for the requested workspace.")

    manifest_df["item_base_folder"] = manifest_df["target_folder"]
    manifest_df = manifest_df.sort_values(
        by=["item_type", "item_name"],
        key=lambda s: s.map(item_sort_bucket) if s.name == "item_type" else s
    ).reset_index(drop=True)

    display(manifest_df[[
        "workspace_name", "item_type", "item_name", "item_id", "status", "target_folder"
    ]])

except Exception as ex:
    print("Cell 6 failed.")
    print(str(ex))
    raise

## CREATE TARGET WORKSPACE ( CELL 7 )

In [ ]:
# =========================================================
# CREATE TARGET WORKSPACE
# =========================================================

workspace_restore_summary = {
    "source_workspace_name": SOURCE_WORKSPACE_NAME,
    "source_workspace_id": workspace_json.get("id", ""),
    "target_workspace_name": TARGET_WORKSPACE_NAME,
    "target_workspace_id": "",
    "workspace_create_status": "",
    "workspace_access_restored": 0,
    "workspace_access_skipped": 0,
    "workspace_access_errors": 0,
    "dashboard_restore_status": "skipped_inventory_only",
    "dashboard_inventory_count": int(len(dashboard_inventory)),
    "restore_utc": datetime.now(timezone.utc).isoformat()
}

created_workspace = create_workspace_from_backup(workspace_json)
target_workspace_id = created_workspace["id"]

workspace_restore_summary["target_workspace_id"] = target_workspace_id
workspace_restore_summary["workspace_create_status"] = "created"

log_message(f"Target workspace created: {TARGET_WORKSPACE_NAME} ({target_workspace_id})")

if RESTORE_WORKSPACE_ACCESS:
    restored_count, skipped_count, access_error_count = restore_workspace_access(target_workspace_id, workspace_access)
    workspace_restore_summary["workspace_access_restored"] = int(restored_count)
    workspace_restore_summary["workspace_access_skipped"] = int(skipped_count)
    workspace_restore_summary["workspace_access_errors"] = int(access_error_count)
else:
    workspace_restore_summary["workspace_access_errors"] = 0
    log_message("Workspace access restore disabled by parameter.")

write_restore_artifact("workspace_restore_summary.json", workspace_restore_summary)
display(pd.DataFrame([workspace_restore_summary]))

## RESTORE ITEMS ( CELL 8 )

In [ ]:
# =========================================================
# RESTORE ITEMS
# =========================================================

restore_rows = []

success_count = 0
skip_count = 0
error_count = 0

restored_semantic_models = {}

for _, row in manifest_df.iterrows():
    item_name = row["item_name"]
    item_type = row["item_type"]
    item_id = row["item_id"]
    item_base_folder = row["item_base_folder"]
    prior_status = row.get("status", "")

    restore_row = {
        "source_workspace_name": SOURCE_WORKSPACE_NAME,
        "source_item_name": item_name,
        "source_item_id": item_id,
        "item_type": item_type,
        "target_workspace_id": target_workspace_id,
        "target_item_name": item_name,
        "target_item_id": "",
        "restore_status": "",
        "dataset_permissions_applied": 0,
        "dataset_permissions_skipped": 0,
        "rls_members_applied": 0,
        "rls_members_skipped": 0,
        "message": "",
        "source_folder": item_base_folder
    }

    try:
        log_message(f"Restoring item: {item_type} | {item_name}")

        if prior_status not in ("exported", "Exported", "skipped_unsupported", "skipped_protected_label"):
            restore_row["restore_status"] = "skipped_not_exported_in_backup"
            restore_row["message"] = f"Backup manifest status was '{prior_status}'"
            restore_rows.append(restore_row)
            skip_count += 1
            continue

        if item_type == "Dashboard":
            restore_row["restore_status"] = "skipped_inventory_only"
            restore_row["message"] = "Dashboard definition not backed up in source notebook."
            restore_rows.append(restore_row)
            skip_count += 1
            continue

        if item_type not in ("SemanticModel", "Report", "PaginatedReport", "DataflowGen1", "Dataflow") and not RESTORE_OTHER_DEFINITION_ITEMS:
            restore_row["restore_status"] = "skipped_by_parameter"
            restore_row["message"] = "Generic definition-based restore disabled."
            restore_rows.append(restore_row)
            skip_count += 1
            continue

        # -------------------------------------------------
        # Semantic Model
        # -------------------------------------------------
        if item_type == "SemanticModel":
            definition = get_definition_from_backup(item_base_folder)
            created = create_semantic_model(
                workspace_id=target_workspace_id,
                display_name=item_name,
                definition=definition,
                description=""
            )

            new_item_id = created.get("id", "")
            restore_row["target_item_id"] = new_item_id
            restore_row["restore_status"] = "restored"

            restored_semantic_models[item_id] = {
                "new_id": new_item_id,
                "new_name": item_name
            }

            log_message(f"Semantic model restored: {item_name} -> {new_item_id}")

            backup_rls = get_backup_rls_role_summary(item_base_folder)
            restored_rls = inspect_restored_semantic_model_roles(
                workspace_id=target_workspace_id,
                semantic_model_id=new_item_id
            )

            if backup_rls["has_backup"]:
                log_message(
                    f"Backup RLS summary for '{item_name}': roles={backup_rls['roles']}, rows={backup_rls['rows']}"
                )
            log_message(
                f"Restored model RLS summary for '{item_name}': roles={restored_rls.get('roles', [])}, "
                f"status={restored_rls.get('status', '')}"
            )

            if backup_rls["has_backup"] and backup_rls["status"] == "ok":
                missing_from_model = sorted(
                    set([str(x).strip().lower() for x in backup_rls["roles"]]) -
                    set([str(x).strip().lower() for x in restored_rls.get("roles", [])])
                )
                if missing_from_model:
                    restore_row["message"] = (
                        "Warning: backup contains RLS roles that do not exist in restored model. "
                        "This means role rules/filters were not present in the restored semantic model definition. "
                        f"Missing roles: {missing_from_model}"
                    )
                    log_message(restore_row["message"])

            if ENABLE_CLIENT_THROTTLING:
                time.sleep(2)

            if RESTORE_DATASET_PERMISSIONS and new_item_id:
                applied, skipped = apply_dataset_permissions(
                    workspace_id=target_workspace_id,
                    dataset_id=new_item_id,
                    item_base_folder=item_base_folder
                )
                restore_row["dataset_permissions_applied"] = int(applied)
                restore_row["dataset_permissions_skipped"] = int(skipped)

            if ENABLE_CLIENT_THROTTLING:
                time.sleep(3)

            if RESTORE_RLS_ROLE_MEMBERS and new_item_id:
                rls_result = restore_semantic_model_rls_members(
                    workspace_id=target_workspace_id,
                    semantic_model_id=new_item_id,
                    semantic_model_name=item_name,
                    item_folder_path=item_base_folder
                )

                restore_row["rls_members_applied"] = int(rls_result.get("restored", 0))
                restore_row["rls_members_skipped"] = int(rls_result.get("skipped", 0))

                rls_status = str(rls_result.get("status", ""))
                rls_errors = int(rls_result.get("errors", 0))
                missing_roles = rls_result.get("missing_roles", [])

                if rls_status and rls_status != "completed":
                    details = [f"RLS status={rls_status}"]
                    if rls_errors:
                        details.append(f"errors={rls_errors}")
                    if missing_roles:
                        details.append(f"missing_roles={missing_roles}")
                    rls_msg = "; ".join(details)

                    if restore_row["message"]:
                        restore_row["message"] = restore_row["message"] + " | " + rls_msg
                    else:
                        restore_row["message"] = rls_msg

                    log_message(f"RLS restore diagnostics for '{item_name}': {rls_msg}")

            restore_rows.append(restore_row)
            success_count += 1
            continue

        # -------------------------------------------------
        # Report
        # -------------------------------------------------
        if item_type == "Report":
            definition = get_definition_from_backup(item_base_folder)
            created = create_report(
                workspace_id=target_workspace_id,
                display_name=item_name,
                definition=definition,
                description=""
            )

            restore_row["target_item_id"] = created.get("id", "")
            restore_row["restore_status"] = "restored"
            restore_rows.append(restore_row)
            success_count += 1
            log_message(f"Report restored: {item_name}")
            continue

        # -------------------------------------------------
        # Paginated Report
        # -------------------------------------------------
        if item_type == "PaginatedReport":
            rdl_path = get_paginated_rdl_path(item_base_folder)
            rdl_bytes = get_file_bytes(rdl_path)

            imported = import_paginated_report(
                workspace_id=target_workspace_id,
                display_name=item_name,
                rdl_bytes=rdl_bytes
            )

            restore_row["target_item_id"] = imported.get("id", "")
            restore_row["restore_status"] = "restored"
            restore_row["message"] = imported.get("importState", "")
            restore_rows.append(restore_row)
            success_count += 1
            log_message(f"Paginated report restored: {item_name}")
            continue

        # -------------------------------------------------
        # Dataflow Gen1
        # -------------------------------------------------
        if item_type == "DataflowGen1":
            print("DATA FLOW GEN 1 LOOP")
            model_json_path = get_dataflow_gen1_model_json_path(item_base_folder)
            model_json_bytes = get_file_bytes(model_json_path)

            imported = import_dataflow_gen1(
                workspace_id=target_workspace_id,
                display_name=item_name,
                model_json_bytes=model_json_bytes
            )

            restore_row["target_item_id"] = imported.get("id", "")
            restore_row["restore_status"] = "restored"

            metadata_json = get_optional_json_if_exists(f"{item_base_folder}/metadata.json")
            datasources_json = get_optional_json_if_exists(f"{item_base_folder}/datasources.json")
            upstream_json = get_optional_json_if_exists(f"{item_base_folder}/upstreamDataflows.json")

            diagnostics = []
            diagnostics.append(f"importState={imported.get('importState', '')}")

            if metadata_json is not None:
                diagnostics.append("metadata.json found")
            if datasources_json is not None:
                diagnostics.append("datasources.json found")
            if upstream_json is not None:
                diagnostics.append("upstreamDataflows.json found")

            restore_row["message"] = "; ".join(diagnostics)

            restore_rows.append(restore_row)
            success_count += 1
            log_message(f"Dataflow Gen1 restored: {item_name}")
            continue

        # -------------------------------------------------
        # Fabric Dataflow (definition-based) if present
        # -------------------------------------------------
        if item_type == "Dataflow":
            definition = get_definition_from_backup(item_base_folder)
            created = create_item_generic(
                workspace_id=target_workspace_id,
                item_type=item_type,
                display_name=item_name,
                definition=definition,
                description=""
            )

            restore_row["target_item_id"] = created.get("id", "")
            restore_row["restore_status"] = "restored"
            restore_rows.append(restore_row)
            success_count += 1
            log_message(f"Fabric Dataflow restored: {item_name}")
            continue

        # -------------------------------------------------
        # Other definition-based item types
        # -------------------------------------------------
        definition = get_definition_from_backup(item_base_folder)
        created = create_item_generic(
            workspace_id=target_workspace_id,
            item_type=item_type,
            display_name=item_name,
            definition=definition,
            description=""
        )

        restore_row["target_item_id"] = created.get("id", "")
        restore_row["restore_status"] = "restored"
        restore_rows.append(restore_row)
        success_count += 1
        log_message(f"Generic item restored: {item_type} | {item_name}")

    except Exception as ex:
        restore_row["restore_status"] = "error"
        restore_row["message"] = str(ex)

        log_message(f"ERROR restoring {item_type} | {item_name}: {ex}")
        log_message(traceback.format_exc())

        restore_rows.append(restore_row)
        error_count += 1
        raise_if_fail_fast(ex)

restore_df = pd.DataFrame(restore_rows)

summary = {
    "source_workspace_name": SOURCE_WORKSPACE_NAME,
    "target_workspace_name": TARGET_WORKSPACE_NAME,
    "target_workspace_id": target_workspace_id,
    "success_count": int(success_count),
    "skip_count": int(skip_count),
    "error_count": int(error_count),
    "dashboard_restore_status": "skipped_inventory_only",
    "restore_utc": datetime.now(timezone.utc).isoformat()
}

write_restore_artifact("item_restore_manifest.json", restore_rows)
write_restore_artifact("item_restore_summary.json", summary)

display(restore_df.sort_values(["item_type", "source_item_name"]))
display(pd.DataFrame([summary]))

print(f"Restore complete. Success={success_count}, Skipped={skip_count}, Errors={error_count}")

## TROUBLE SHOOTING and SUMMARY REPORT ( CELL 9 )

In [ ]:
# =========================================================
# TROUBLESHOOTING / SUMMARY REPORT
# =========================================================
import io
import csv

summary_report_df = restore_df.copy()

preferred_cols = [
    "source_workspace_name",
    "source_item_name",
    "source_item_id",
    "item_type",
    "target_workspace_id",
    "target_item_name",
    "target_item_id",
    "restore_status",
    "dataset_permissions_applied",
    "dataset_permissions_skipped",
    "rls_members_applied",
    "rls_members_skipped",
    "message",
    "source_folder"
]

summary_report_df = summary_report_df[[c for c in preferred_cols if c in summary_report_df.columns]]

def _safe_cell_value(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v) if not isinstance(v, (str, int, float, bool)) else v

# Build plain Python records without pandas serializers
records = []
column_names = list(summary_report_df.columns)

for row in summary_report_df.itertuples(index=False, name=None):
    rec = {}
    for i, col in enumerate(column_names):
        rec[col] = _safe_cell_value(row[i])
    records.append(rec)

# JSON report
json_text = json.dumps(records, indent=2, ensure_ascii=False, default=str)
write_text(
    f"{RESTORE_LOG_ROOT}/restore_report.json",
    json_text,
    overwrite=True
)

# CSV report using stdlib only
csv_buffer = io.StringIO()
writer = csv.writer(csv_buffer)
writer.writerow(column_names)

for rec in records:
    writer.writerow([rec.get(col, "") for col in column_names])

write_text(
    f"{RESTORE_LOG_ROOT}/restore_report.csv",
    csv_buffer.getvalue(),
    overwrite=True
)

print("=========================================================")
print("RESTORE SUMMARY")
print("=========================================================")
print(f"Source workspace   : {SOURCE_WORKSPACE_NAME}")
print(f"Target workspace   : {TARGET_WORKSPACE_NAME}")
print(f"Target workspace id: {target_workspace_id}")
print(f"Success            : {success_count}")
print(f"Skipped            : {skip_count}")
print(f"Errors             : {error_count}")
print("---------------------------------------------------------")
print("Skipped dashboards are expected because backup captured")
print("dashboard inventory only, not a restorable definition.")
print("---------------------------------------------------------")
print(f"Restore log folder : {RESTORE_LOG_ROOT}")
print("=========================================================")

display(summary_report_df.sort_values(["restore_status", "item_type", "source_item_name"]))

## OPTIONAL : QUICK ERROR FILTER ( CELL 10 )

In [ ]:
# =========================================================
# OPTIONAL: QUICK ERROR FILTER
# =========================================================

error_df = summary_report_df[summary_report_df["restore_status"] == "error"].copy()
skip_df = summary_report_df[
    summary_report_df["restore_status"].astype(str).str.startswith("skipped", na=False)
].copy()

print(f"Errors found : {len(error_df)}")
print(f"Skips found  : {len(skip_df)}")

if len(error_df) > 0:
    display(error_df[["item_type", "source_item_name", "message"]])

if len(skip_df) > 0:
    display(skip_df[["item_type", "source_item_name", "restore_status", "message"]])